## Получение списка дат экспираций

# Тестирование сохранения данных опционов в БД

Этот notebook используется для проверки работы периодического сохранения данных из WebSocket в SQLite базу данных.

## Инструкция по тестированию:

1. **Запустите приложение в Docker:**
   ```bash
   docker compose up --build
   ```

2. **Дождитесь подключения WebSocket и начала сбора данных:**
   - В логах должно появиться сообщение "✅ WebSocket connected"
   - Должно появиться "Периодическое сохранение данных в БД запущено"
   - Подождите минимум 5-10 минут, чтобы накопились данные

3. **Если БД находится в Docker контейнере, скопируйте её локально:**
   ```bash
   docker cp option-telegram-bot:/app/data/options.db ./data/options.db
   ```
   Или если БД монтируется как volume, она уже будет доступна локально.

4. **Запустите ячейки ниже для проверки данных**

## Строка подключения к БД:

**Локально:** `./data/options.db` (относительно корня проекта)
**В Docker:** `/app/data/options.db` (внутри контейнера)

In [ ]:
from pybit.unified_trading import HTTP

def get_option_expirations_pybit():
    session = HTTP(testnet=False)
    
    try:
        response = session.get_instruments_info(
            category="option",
            status="Trading",
            baseCoin="BTC",
            settleCoin="USDT",
            limit=1000
        )
        
        if response['retCode'] == 0:
            expiry_dates = set()
            for instrument in response['result']['list']:
                if instrument['settleCoin'] == 'USDT':
                    expiry_dates.add(instrument['deliveryTime'])
            
            return sorted(list(expiry_dates))
        else:
            print(f"Ошибка: {response['retMsg']}")
            return []
    except Exception as e:
        print(f"Исключение: {e}")
        return []

In [ ]:
get_option_expirations_pybit()

## Тестирование

In [ ]:
import sys

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path
from datetime import datetime, timedelta
from IPython.display import display

# Путь к базе данных
# В Docker контейнере база находится в /app/data/options.db
# Локально - в ./data/options.db относительно корня проекта
project_root = Path.cwd()
db_path = project_root / "data" / "options.db"

print(f"Путь к БД: {db_path}")
print(f"БД существует: {db_path.exists()}")

# Строка подключения для SQLite (просто путь к файлу)
connection_string = str(db_path)
print(f"\nСтрока подключения: {connection_string}")

Путь к БД: /media/alex/afd38d35-3963-4272-9b6c-973ab1698dd5/Боты/Bot_Option_cursor/data/options.db
БД существует: True

Строка подключения: /media/alex/afd38d35-3963-4272-9b6c-973ab1698dd5/Боты/Bot_Option_cursor/data/options.db


In [4]:
# Подключение к БД
conn = sqlite3.connect(str(db_path))
conn.row_factory = sqlite3.Row  # Для доступа к колонкам по имени

# Проверка таблиц
cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
tables = cursor.fetchall()
print("Таблицы в БД:")
for table in tables:
    print(f"  - {table[0]}")

Таблицы в БД:
  - option_history
  - sqlite_sequence
  - underlying_history
  - iv_history
  - support_resistance_levels
  - agent_signals
  - signal_results
  - strategy_thresholds
  - gex_presets


In [5]:
# 1. Проверка количества записей в option_history
query = "SELECT COUNT(*) as count FROM option_history"
df = pd.read_sql_query(query, conn)
print("Количество записей в option_history:")
print(df)

Количество записей в option_history:
    count
0  569105


In [6]:
# 2. Последние 10 записей из option_history
query = """
SELECT 
  *
FROM option_history
"""
df = pd.read_sql_query(query, conn)
print("Последние 10 записей:")
display(df.head())

Последние 10 записей:


,id,symbol,date_data_collection,expiration_date,underlying_ticker,days_to_expiration,ask_price,bid_price,mark_price,iv,...,bid_iv,mark_iv,delta,gamma,vega,theta,volume_24h,open_interest,underlying_price,is_otm
0,1,BTC-18JAN26-100000-C-USDT,2026-01-15 18:15:00,2026-01-18,BTC,3,255.0,250.0,257.085872,0.3621,...,0.3586,0.3621,0.159771,0.000078,20.835159,-131.828418,1.37,1.35,96809.825157,1
1,2,BTC-18JAN26-99000-C-USDT,2026-01-15 18:15:00,2026-01-18,BTC,3,415.0,410.0,418.468838,0.3504,...,0.3471,0.3504,0.240183,0.000104,26.656605,-163.207051,3.78,3.56,96809.825157,1
2,3,BTC-18JAN26-98500-C-USDT,2026-01-15 18:15:00,2026-01-18,BTC,3,530.0,525.0,532.209194,0.3455,...,0.3429,0.3455,0.290943,0.000116,29.384175,-177.384824,0.46,0.05,96809.825157,1
3,4,BTC-18JAN26-98000-C-USDT,2026-01-15 18:15:00,2026-01-18,BTC,3,690.0,665.0,673.801162,0.3414,...,0.3385,0.3414,0.348544,0.000126,31.699277,-189.070821,1.30,0.50,96809.825157,1
4,5,BTC-18JAN26-97500-C-USDT,2026-01-15 18:15:00,2026-01-18,BTC,3,855.0,840.0,847.477251,0.3381,...,0.3358,0.3381,0.412009,0.000134,33.358983,-197.079283,2.51,0.75,96809.825157,1


In [12]:
df['is_otm'].value_counts()

is_otm
1    569105
Name: count, dtype: int64

In [11]:
df[df['is_otm'] == False]

,id,symbol,date_data_collection,expiration_date,underlying_ticker,days_to_expiration,ask_price,bid_price,mark_price,iv,...,bid_iv,mark_iv,delta,gamma,vega,theta,volume_24h,open_interest,underlying_price,is_otm


In [ ]:
query = """
SELECT 
    *
FROM option_history
ORDER BY timestamp DESC
LIMIT 3
"""
df = pd.read_sql_query(query, conn)
print("Последние 10 записей:")
print(df.to_string())

In [ ]:
# 3. Уникальные символы в БД
query = """
SELECT DISTINCT symbol, COUNT(*) as count
FROM option_history
GROUP BY symbol
ORDER BY count DESC
"""
df = pd.read_sql_query(query, conn)
print("Уникальные символы и количество записей:")
print(df.to_string())

In [ ]:
# 4. Проверка временных интервалов сохранения (должны быть кратны 5 минутам)
query = """
SELECT 
    timestamp,
    strftime('%M', timestamp) as minute,
    COUNT(*) as count
FROM option_history
GROUP BY timestamp
ORDER BY timestamp DESC
LIMIT 20
"""
df = pd.read_sql_query(query, conn)
print("Временные метки сохранения (последние 20):")
print(df.to_string())

# Проверяем, что минуты кратны 5
if len(df) > 0:
    minutes = df['minute'].astype(int)
    not_aligned = minutes[minutes % 5 != 0]
    if len(not_aligned) > 0:
        print(f"\n⚠️ ВНИМАНИЕ: Найдены записи не выровненные по 5 минутам: {not_aligned.tolist()}")
    else:
        print("\n✅ Все временные метки выровнены по 5-минутным интервалам")

In [ ]:
# 5. История для конкретного символа (если есть данные)
query = """
SELECT symbol
FROM option_history
LIMIT 1
"""
df_symbol = pd.read_sql_query(query, conn)

if len(df_symbol) > 0:
    test_symbol = df_symbol['symbol'].iloc[0]
    print(f"Анализ истории для символа: {test_symbol}\n")
    
    query_history = """
    SELECT 
        timestamp,
        ask_price,
        bid_price,
        mark_price,
        iv,
        delta,
        underlying_price
    FROM option_history
    WHERE symbol = ?
    ORDER BY timestamp ASC
    """
    df_history = pd.read_sql_query(query_history, conn, params=(test_symbol,))
    print(f"Количество записей: {len(df_history)}")
    print("\nПервые 5 записей:")
    print(df_history.head().to_string())
    print("\nПоследние 5 записей:")
    print(df_history.tail().to_string())
else:
    print("В БД пока нет данных")

In [ ]:
# 6. Статистика по underlying_history
query = """
SELECT 
    symbol,
    COUNT(*) as count,
    MIN(timestamp) as first_record,
    MAX(timestamp) as last_record
FROM underlying_history
GROUP BY symbol
"""
df = pd.read_sql_query(query, conn)
print("Статистика по базовым активам:")
print(df.to_string())

In [ ]:
# 7. Проверка IV истории
query = """
SELECT 
    symbol,
    COUNT(*) as count,
    AVG(iv) as avg_iv,
    MIN(iv) as min_iv,
    MAX(iv) as max_iv
FROM iv_history
WHERE iv IS NOT NULL
GROUP BY symbol
ORDER BY count DESC
LIMIT 10
"""
df = pd.read_sql_query(query, conn)
print("Статистика IV по символам:")
print(df.to_string())

In [ ]:
# Закрываем соединение
conn.close()
print("Соединение с БД закрыто")